# IJCAI 2027 — Ý tưởng 1: Kiểm định Tính Cục bộ Không gian (Spatial Locality) trên 20 Mô hình (2020–2024)

Notebook này tự động:
1. **Mount Google Drive** và **pull trực tiếp code + data cần thiết từ GitHub (`idea1-colab`) về Google Drive**.
2. Tải 8 kho mã nguồn gốc (commit cố định) + trọng số công bố của **18 mô hình** (gồm Diffusion `SingularTrajectory` CVPR'24, Equivariant GNN `EqMotion` CVPR'23, Transformer `ET-AgentFormer` ICCV'23, `PECNet`, `Graph-TERN`, `Social-Implicit`, `SGCN`, `DMRGCN`, `Social-STGCNN`, ...) kèm **2 mô hình bất biến thứ tự Width-1 (`STGCNN-F`, `DMRGCN-F`)** đóng gói sẵn trong repo.
3. Chạy phép thử **xóa từng người đi bộ (Co-occurrence Graph Coloring)** và lưu tiến độ trực tiếp vào Google Drive (nếu rớt mạng chỉ cần **Run all** lại là chạy tiếp phần còn dở).

### Hướng dẫn chạy:
1. Vào **Runtime → Change runtime type** → chọn **GPU** (T4 / L4 / A100 / G4 đều chạy tốt).
2. Bấm **Runtime → Run all**.
3. Khi chạy xong Ô 4, tải file nhẹ `idea1_summary_for_paper.zip` (~vài trăm KB) trong thư mục `MyDrive/TTC_STGCNN_IJCAI2027/` về máy tính và gửi lại vào workspace.

In [ ]:
# Ô 1: Mount Google Drive & Pull code + data từ GitHub về thẳng Google Drive
import os, sys, subprocess, shutil, glob, json
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/Thanh-2312/TTC-STGCNN.git'
BRANCH = 'idea1-colab'
DRIVE_BASE = '/content/drive/MyDrive/TTC_STGCNN_IJCAI2027'
REPO_DIR = os.path.join(DRIVE_BASE, 'repo')
WORK_DIR = os.path.join(DRIVE_BASE, 'idea1_work')
os.makedirs(DRIVE_BASE, exist_ok=True)

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    print(f'Đang clone {REPO_URL} (nhánh {BRANCH}) về Google Drive...')
    subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Đang cập nhật (git pull) nhánh {BRANCH} trên Google Drive...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)

RUNNER = os.path.join(REPO_DIR, 'colab', 'idea1', 'run_loc.py')
ANALYZER = os.path.join(REPO_DIR, 'colab', 'idea1', 'analyze_loc.py')
assert os.path.exists(RUNNER), f'Không tìm thấy {RUNNER}'

import torch
print('Repo trên Drive:', REPO_DIR)
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU', '| CPU cores:', os.cpu_count())
assert torch.cuda.is_available(), 'Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> chọn GPU.'

# Kiểm tra nhanh tính toàn vẹn logic ngay trên Colab trước khi chạy
subprocess.run([sys.executable, ANALYZER, '--selftest'], check=True)

def run_idea1(**kw):
    env = dict(os.environ, **{k: str(v) for k, v in kw.items()})
    p = subprocess.Popen([sys.executable, '-u', RUNNER], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    return p.wait()

In [ ]:
# Ô 2: Chạy thử nhanh (Smoke Test ~5-10 phút trên tập eth, 1 màu xóa) để xác nhận cả 20 mô hình chạy trơn tru
SMOKE_DIR = '/content/idea1_smoke'
shutil.rmtree(SMOKE_DIR, ignore_errors=True)
shutil.rmtree('/content/loc_pert_smoke', ignore_errors=True)
workers = max(4, min(16, (os.cpu_count() or 4)))
rc = run_idea1(LOC_WORK=SMOKE_DIR, LOC_PERT='/content/loc_pert_smoke', LOC_DS='eth', LOC_COLORS=1, LOC_WORKERS=workers)

failed_smoke = json.load(open(os.path.join(SMOKE_DIR, 'failed.json'))) if os.path.exists(os.path.join(SMOKE_DIR, 'failed.json')) else ['missing']
print('\n=== KẾT QUẢ SMOKE TEST ===')
print('Số job lỗi:', len(failed_smoke), failed_smoke)
if failed_smoke:
    for key in failed_smoke[:5]:
        lf = os.path.join(SMOKE_DIR, 'logs', f'{key}.txt')
        if os.path.exists(lf):
            print(f'\n--- Log lỗi {key} ---')
            print(''.join(open(lf).readlines()[-15:]))
if os.path.exists(os.path.join(SMOKE_DIR, 'results.md')):
    print('\n' + open(os.path.join(SMOKE_DIR, 'results.md')).read())

In [ ]:
# Ô 3: Chạy ĐẦY ĐỦ trên cả 5 tập dữ liệu ETH/UCY — lưu trực tiếp vào Google Drive (hỗ trợ chạy tiếp nếu đứt kết nối)
workers = max(4, min(24, (os.cpu_count() or 4)))
print(f'Bắt đầu chạy đầy đủ vào {WORK_DIR} với {workers} tiến trình song song...')
run_idea1(LOC_WORK=WORK_DIR, LOC_PERT='/content/loc_pert_full', LOC_COLORS=12, LOC_WORKERS=workers)

res_md = os.path.join(WORK_DIR, 'results.md')
if os.path.exists(res_md):
    print('\n' + '='*80 + '\n')
    print(open(res_md, encoding='utf-8').read())

res_png = os.path.join(WORK_DIR, 'results_overview.png')
if os.path.exists(res_png):
    from IPython.display import Image, display
    display(Image(filename=res_png))

In [ ]:
# Ô 4: Đóng gói kết quả gọn nhẹ trên Google Drive để tải về máy tính đưa vào bài báo IJCAI 2027
import zipfile

summary_zip = os.path.join(DRIVE_BASE, 'idea1_summary_for_paper.zip')
with zipfile.ZipFile(summary_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for fn in ('results.json', 'results.md', 'results_overview.pdf', 'results_overview.png', 'schedule.json', 'commits.json', 'failed.json'):
        fp = os.path.join(WORK_DIR, fn)
        if os.path.exists(fp):
            z.write(fp, fn)

done_json = len(glob.glob(os.path.join(WORK_DIR, 'out', '*', '*', '*.json')))
print(f'Đã hoàn thành {done_json} job files.')
print('File kết quả gọn nhẹ (tải file này về máy):', summary_zip, f'({os.path.getsize(summary_zip)/1024:.1f} KB)')